# EMG — Line In Left: bandpass + a mains-ratio filter selector

Reads **only** `emg_line_L.csv` from a recording folder you pick.

**What it does**

1. **Load** `emg_line_L.csv` from the chosen folder (nothing else is touched).
2. **Bandpass 5–500 Hz** — Butterworth, zero-phase `sosfiltfilt`. This is the
   reference signal everything below is measured on.
3. **Mains ratio** — the proportion of spectral power in the **49–51 Hz**
   window to the **rest** of the 5–500 Hz band. One number that says how badly
   the recording is contaminated by 50 Hz powerline hum.
4. **Pick a powerline filter from that ratio — two rules:**
   * `ratio < 10%`  → **normalised feed-forward comb**  `y[n] = x[n] − x[n−M]`
   * `ratio ≥ 10%`  → **normalised IIR comb, order 5**  (r = 0.9)
5. **Apply it and plot** — interactive Plotly time and spectrum figures, raw →
   bandpass → filtered, with the 49–51 Hz window highlighted.

The threshold is on the **`ratio`** metric (`P(49–51 Hz)/P(rest)`), so a
recording printed as `ratio = 0.15` counts as **15 %** — above the 10 % line.

Run the cells top to bottom, then use the widget. The final cell sweeps every
recording so you can see where the 10 % line splits them.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import signal as sig

import plotly.graph_objects as go
from plotly.subplots import make_subplots

import ipywidgets as widgets
from IPython.display import display, clear_output

RECORDINGS_DIR = Path("recordings")
CHANNEL_FILE   = "emg_line_L.csv"   # this notebook reads this file and nothing else

# ── analysis band & mains window ──────────────────────────────────────────────
BAND        = (5.0, 500.0)   # Hz — EMG band, also the bandpass passband
MAINS_WIN   = (49.0, 51.0)   # Hz — the "50 Hz" window whose power we measure
HARMONIC_HZ = 50.0           # Hz — nominal mains fundamental (harmonics at 100,150…)
HARM_BW     = 1.0            # Hz — ± window used when summing harmonic power

# ── decision rule ─────────────────────────────────────────────────────────────
RATIO_THRESHOLD = 0.10       # 10% — split point on the mains ratio P(49–51)/P(rest)

# ── comb-filter constants (match emg_line_L_bandpass.ipynb / *_verification.py) ─
F0_MAINS       = 49.97328    # Hz — measured mains fundamental, sets the comb delay M
R_M            = 0.9         # IIR comb feedback coefficient
IIR_COMB_ORDER = 5           # cascade order of the IIR comb

## Loading, filtering, and the FFT

`fs` is recovered from the timestamps (median sample spacing — immune to a single
glitched row). The bandpass is built as second-order sections: 5 Hz at 44.1 kHz
is a tiny normalised frequency where `(b, a)` coefficients lose precision but SOS
does not.

In [ ]:
def find_folders(root: Path = RECORDINGS_DIR):
    """Every folder under *root* holding an emg_line_L.csv, searched recursively
    (so recordings/extras/… shows up too). Returns paths relative to *root*."""
    if not root.exists():
        return []
    return sorted(p.parent.relative_to(root).as_posix()
                  for p in root.rglob(CHANNEL_FILE))


def load_line_L(folder: str):
    """Read emg_line_L.csv from *folder* → (time, signal, fs)."""
    path = RECORDINGS_DIR / folder / CHANNEL_FILE
    if not path.exists():
        raise FileNotFoundError(f"{CHANNEL_FILE} not found in {path.parent}")
    df = pd.read_csv(path)
    t = df["time"].to_numpy(dtype=float)
    x = df["data"].to_numpy(dtype=float)
    fs = 1.0 / float(np.median(np.diff(t)))
    return t, x, fs


def bandpass_filter(x, fs, low_hz=BAND[0], high_hz=BAND[1], order=4):
    """Zero-phase Butterworth bandpass, SOS form for numerical stability."""
    nyq = fs / 2.0
    if high_hz >= nyq:
        high_hz = nyq * 0.99
    sos = sig.butter(order, [low_hz, high_hz], btype="band", fs=fs, output="sos")
    return sig.sosfiltfilt(sos, x)


def rfft_single_sided(x, fs):
    """Single-sided amplitude spectrum → (freqs, magnitude)."""
    N = len(x)
    freqs = np.fft.rfftfreq(N, d=1.0 / fs)
    mag = np.abs(np.fft.rfft(x)) * (2.0 / N)
    mag[0] /= 2.0
    if N % 2 == 0:
        mag[-1] /= 2.0
    return freqs, mag

## The mains ratio — the number that drives the decision

Power per FFT bin is $|X[k]|^2$; by Parseval, summing it over a frequency band
gives that band's energy. The headline metric is exactly the proportion you
asked for:

$$\text{ratio} \;=\; \frac{P(49\text{–}51\,\text{Hz})}{P(\text{rest of }5\text{–}500\,\text{Hz})}$$

Two design points worth stating out loud:

* **Measured on the band-passed signal.** "The rest of the spectrum" is the
  5–500 Hz EMG band with the mains window taken out — *not* the raw signal,
  whose sub-5 Hz baseline wander would swamp the denominator and hide the hum.
* **Use the raw FFT, not Welch.** The rfft bin width here is $f_s/N \approx
  0.2$ Hz, so the 2 Hz mains window holds ~10 bins. A coarse Welch estimate
  (~2.7 Hz bins) is *wider* than the window and collapses the ratio to zero.

`ratio_harm` repeats the measurement over every mains harmonic (50, 100, 150 …
Hz); it is a secondary diagnostic — the fundamental-only `ratio` drives the rule.

In [ ]:
def mains_metrics(x, fs, band=BAND, mains=MAINS_WIN,
                  harmonic_hz=HARMONIC_HZ, harm_bw=HARM_BW,
                  harmonics_max=None):
    """Mains-contamination metrics from the amplitude spectrum of *x*.

    Returns a dict:
      ratio       P(49–51 Hz) / P(rest of band)      ← the requested proportion
      frac        P(49–51 Hz) / P(band)              same info, as a 0–1 fraction
      ratio_harm  P(all harmonics) / P(rest)         fundamental + 100,150,… Hz
      frac_harm   P(all harmonics) / P(band)
      p_band, p_mains, p_harm                        raw band energies
    """
    if harmonics_max is None:
        harmonics_max = band[1]
    N = len(x)
    freqs = np.fft.rfftfreq(N, d=1.0 / fs)
    power = np.abs(np.fft.rfft(x)) ** 2               # ∝ energy per bin (Parseval)

    inband  = (freqs >= band[0]) & (freqs <= band[1])
    ismains = inband & (freqs >= mains[0]) & (freqs <= mains[1])

    isharm = np.zeros(freqs.shape, dtype=bool)        # every k·50 Hz ± harm_bw
    k = 1
    while k * harmonic_hz <= harmonics_max:
        fc = k * harmonic_hz
        isharm |= (freqs >= fc - harm_bw) & (freqs <= fc + harm_bw)
        k += 1
    isharm &= inband

    p_band  = power[inband].sum()
    p_mains = power[ismains].sum()
    p_harm  = power[isharm].sum()
    p_rest    = p_band - p_mains
    p_rest_h  = p_band - p_harm

    return dict(
        ratio      = p_mains / p_rest   if p_rest   > 0 else np.inf,
        frac       = p_mains / p_band   if p_band   > 0 else 0.0,
        ratio_harm = p_harm  / p_rest_h if p_rest_h > 0 else np.inf,
        frac_harm  = p_harm  / p_band   if p_band   > 0 else 0.0,
        p_band=p_band, p_mains=p_mains, p_harm=p_harm,
    )

## The two comb filters

Both are ported unchanged from `emg_line_L_bandpass.ipynb`, so results here line
up with the comb studies in this repo. Each is applied **zero-phase** (`filtfilt`)
to the band-passed reference; an order-N cascade is N repeated `filtfilt` passes,
never a single degree-N·M polynomial (whose N-fold poles would sit on the unit
circle).

* **Feed-forward comb** — `H(z) = 1 − z⁻ᴹ`, `M = round(fs/f0)`. Nulls every
  harmonic of `f0` in one pass; wide notches, passband follows `|sin|`.
* **IIR comb** — `H(z) = (1 − z⁻ᴹ)/(1 − r·z⁻ᴹ)`. Poles just inside the zeros →
  sharp, narrow notches and a flat passband.

`normalize=True` scales the numerator so one section peaks at exactly 1.0 — notch
depth and position are untouched, only the passband boost (×2 feed-forward,
×2/(1+r) IIR) is removed, so the filter never inflates the signal.

In [ ]:
def comb_section(fs, kind, f0=F0_MAINS, r_m=R_M, normalize=True):
    """(b, a) for ONE comb section. kind is "ff" (feed-forward) or "iir"."""
    M = int(round(fs / f0))
    b = np.zeros(M + 1)
    b[0], b[M] = 1.0, -1.0
    if kind == "ff":
        a = np.array([1.0])
        if normalize:
            b = b * 0.5
    else:
        a = np.zeros(M + 1)
        a[0], a[M] = 1.0, -r_m
        if normalize:
            b = b * (1.0 + r_m) / 2.0
    return b, a


def comb_filter(x, fs, kind, order=1, f0=F0_MAINS, r_m=R_M, normalize=True):
    """Apply a comb of the given cascade order, zero-phase."""
    b, a = comb_section(fs, kind, f0, r_m, normalize)
    y = x.astype(float)
    for _ in range(order):
        y = sig.filtfilt(b, a, y)
    return y

## The decision rule — two rules

$$
\text{filter} =
\begin{cases}
\textbf{normalised feed-forward comb} & \text{ratio} < 10\% \\[2pt]
\textbf{normalised IIR comb, order 5} & \text{ratio} \ge 10\%
\end{cases}
$$

The threshold is `RATIO_THRESHOLD = 0.10` on `ratio = P(49–51 Hz)/P(rest)` — set
in the constants cell, change it in one place. To split on the *fraction of the
whole band* instead (the `% of band` figure), compare `m["frac"]` rather than
`m["ratio"]`.

In [ ]:
FILTER_LABELS = {
    "ff_comb":   "Normalised feed-forward comb  y[n]=x[n]−x[n−M]",
    "iir_comb5": f"Normalised IIR comb, order {IIR_COMB_ORDER}  (r={R_M})",
}
REC_COLORS = {
    "ff_comb":   "#2E7D32",   # green — low mains, gentle comb
    "iir_comb5": "#8E0000",   # red   — high mains, sharp order-5 comb
}


def recommend_filter(ratio, threshold=RATIO_THRESHOLD):
    """Two rules: below the threshold → feed-forward comb, at/above → IIR comb o5."""
    key = "ff_comb" if ratio < threshold else "iir_comb5"
    return key, FILTER_LABELS[key]


def apply_filter(x, fs, key):
    """Run the recommended comb on *x* (the band-passed reference)."""
    if key == "ff_comb":
        return comb_filter(x, fs, "ff", order=1, normalize=True)
    return comb_filter(x, fs, "iir", order=IIR_COMB_ORDER, r_m=R_M, normalize=True)

## Plotting helper

Long recordings (200 k+ samples) are decimated for the time plots with a
peak-preserving envelope: each screen bin keeps its true min and max sample, so
spikes are never hidden while pan/zoom stays responsive.

In [ ]:
def envelope(t, x, n_bins=6000):
    """Peak-preserving decimation: keep the real min & max sample of each bin."""
    n = len(x)
    if n <= 2 * n_bins:
        return t, x
    bin_len = n // n_bins
    usable = bin_len * n_bins
    idx = np.arange(usable).reshape(n_bins, bin_len)
    xb = x[:usable].reshape(n_bins, bin_len)
    rows = np.arange(n_bins)
    keep = np.union1d(idx[rows, np.argmin(xb, axis=1)],
                      idx[rows, np.argmax(xb, axis=1)])
    if usable < n:
        keep = np.union1d(keep, np.arange(usable, n))
    return t[keep], x[keep]

## Main analysis — pick a folder, get the ratio, the chosen filter, and plots

In [ ]:
def analyze(folder, low_hz=BAND[0], high_hz=BAND[1], bp_order=4,
            fft_max_hz=600.0, log_y=False):
    """Bandpass emg_line_L.csv, pick a comb from the mains ratio, apply it, plot."""

    t, raw, fs = load_line_L(folder)
    bp = bandpass_filter(raw, fs, low_hz, high_hz, bp_order)

    m = mains_metrics(bp, fs, band=(low_hz, high_hz))
    key, label = recommend_filter(m["ratio"])
    color = REC_COLORS[key]
    flt = apply_filter(bp, fs, key)

    side = "<" if m["ratio"] < RATIO_THRESHOLD else "≥"

    # ── text summary ──────────────────────────────────────────────────────────
    print(f"{folder}/{CHANNEL_FILE}")
    print(f"  fs = {fs:,.1f} Hz   {len(raw):,} samples   {t[-1] - t[0]:.3f} s")
    print(f"  bandpass {low_hz:g}–{high_hz:g} Hz  (Butterworth order {bp_order}, zero-phase)\n")
    print(f"  mains ratio  P(49–51 Hz) / P(rest)   = {m['ratio']:.4f}  "
          f"= {m['ratio']*100:.1f}%   ({m['frac']*100:.1f}% of the whole 5–500 Hz band)")
    print(f"  with harmonics (50,100,…,500 Hz)     = {m['ratio_harm']:.4f}  "
          f"= {m['ratio_harm']*100:.1f}%")
    print(f"\n  rule:  {m['ratio']*100:.1f}%  {side}  {RATIO_THRESHOLD*100:g}%  →  {label}")

    # ── Fig 1: raw → bandpass → filtered, time domain ─────────────────────────
    t_r, y_r = envelope(t, raw)
    t_b, y_b = envelope(t, bp)
    t_f, y_f = envelope(t, flt)
    fig1 = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.07,
                         subplot_titles=["Raw",
                                         f"Bandpass {low_hz:g}–{high_hz:g} Hz",
                                         label])
    fig1.add_trace(go.Scattergl(x=t_r, y=y_r, name="Raw",
                                line=dict(width=0.7, color="#1565C0")), row=1, col=1)
    fig1.add_trace(go.Scattergl(x=t_b, y=y_b, name="Bandpass",
                                line=dict(width=0.7, color="#E65100")), row=2, col=1)
    fig1.add_trace(go.Scattergl(x=t_f, y=y_f, name="Filtered",
                                line=dict(width=0.7, color=color)), row=3, col=1)
    for r in (1, 2, 3):
        fig1.update_yaxes(title_text="Amplitude (V)", row=r, col=1)
    fig1.update_xaxes(title_text="Time (s)", row=3, col=1)
    fig1.update_layout(title_text=f"Fig 1 — {folder} — raw → bandpass → filtered (time)",
                       height=720, template="plotly_white", hovermode="x unified",
                       legend=dict(orientation="h", y=1.06, x=0.5, xanchor="center"))
    fig1.show()

    # ── Fig 2: spectrum, bandpass vs filtered, mains window highlighted ───────
    fr, mag_bp  = rfft_single_sided(bp,  fs)
    _,  mag_flt = rfft_single_sided(flt, fs)
    fm = fr <= fft_max_hz
    fig2 = go.Figure()
    fig2.add_trace(go.Scattergl(x=fr[fm], y=mag_bp[fm], name="Bandpass",
                                line=dict(width=1.0, color="#E65100")))
    fig2.add_trace(go.Scattergl(x=fr[fm], y=mag_flt[fm], name="Filtered",
                                line=dict(width=1.2, color=color)))
    fig2.add_vrect(x0=MAINS_WIN[0], x1=MAINS_WIN[1],
                   fillcolor="rgba(180,0,0,0.12)", line_width=0,
                   annotation_text="49–51 Hz", annotation_position="top left",
                   annotation_font_size=10, annotation_font_color="rgba(150,0,0,0.9)")
    for k in range(1, int(fft_max_hz // HARMONIC_HZ) + 1):
        fig2.add_vline(x=k * HARMONIC_HZ,
                       line=dict(color="rgba(180,0,0,0.25)", width=1, dash="dot"))
    fig2.add_annotation(
        xref="paper", yref="paper", x=0.99, y=0.97, showarrow=False,
        align="right", bordercolor=color, borderwidth=1.5, borderpad=6,
        bgcolor="rgba(255,255,255,0.85)", font=dict(size=12, color=color),
        text=(f"<b>ratio = {m['ratio']*100:.1f}%</b>  {side} {RATIO_THRESHOLD*100:g}%<br>→ {label}"))
    fig2.update_layout(
        title_text=f"Fig 2 — {folder} — spectrum: bandpass vs filtered",
        xaxis_title="Frequency (Hz)", yaxis_title="Magnitude (V)",
        yaxis_type="log" if log_y else "linear",
        height=460, template="plotly_white", hovermode="x unified",
        legend=dict(orientation="h", y=1.10, x=0.5, xanchor="center"))
    fig2.show()

    print("\nDone.")
    return m

## Widget — choose a folder and run

In [ ]:
clear_output(wait=True)

folders = find_folders()

if not folders:
    print(f"No folder under '{RECORDINGS_DIR}/' contains {CHANNEL_FILE}.")
else:
    def _num(w_cls, value, desc, width="230px", **kw):
        return w_cls(value=value, description=desc,
                     style={"description_width": "initial"},
                     layout=widgets.Layout(width=width), **kw)

    _w_folder = widgets.Dropdown(
        options=folders, value=folders[0], description="Recording:",
        style={"description_width": "initial"},
        layout=widgets.Layout(width="520px"))
    _w_low   = _num(widgets.BoundedFloatText, 5.0,   "Low cutoff (Hz):", "220px",
                    min=0.1, max=2000.0, step=1.0)
    _w_high  = _num(widgets.BoundedFloatText, 500.0, "High cutoff (Hz):", "220px",
                    min=1.0, max=20000.0, step=10.0)
    _w_bpord = _num(widgets.BoundedIntText,   4,     "BP order:", "150px",
                    min=1, max=8, step=1)
    _w_fft   = _num(widgets.BoundedFloatText, 600.0, "FFT max (Hz):", "210px",
                    min=10.0, max=24000.0, step=50.0)
    _w_log   = widgets.Checkbox(value=False, description="Log magnitude axis",
                                indent=False, layout=widgets.Layout(width="200px"))
    _w_run   = widgets.Button(description="▶  Bandpass, choose comb & plot",
                              button_style="primary",
                              layout=widgets.Layout(width="260px", height="38px"))
    _w_out   = widgets.Output()

    def _on_run(b):
        with _w_out:
            clear_output(wait=True)
            try:
                analyze(_w_folder.value,
                        low_hz=float(_w_low.value), high_hz=float(_w_high.value),
                        bp_order=int(_w_bpord.value),
                        fft_max_hz=float(_w_fft.value), log_y=bool(_w_log.value))
            except Exception as exc:
                print(f"{type(exc).__name__}: {exc}")

    _w_run.on_click(_on_run)

    display(widgets.VBox([
        widgets.HTML(f"<b>Pick a recording — only <code>{CHANNEL_FILE}</code> is read.</b>"),
        _w_folder,
        widgets.HBox([_w_low, _w_high, _w_bpord]),
        widgets.HBox([_w_fft, _w_log]),
        _w_run,
        _w_out,
    ]))

## Where the 10 % line falls — the ratio across every recording

Bandpasses every `emg_line_L.csv`, computes the mains ratio, and plots them
sorted and coloured by which of the two combs each one gets. The dashed line is
the 10 % threshold — move `RATIO_THRESHOLD` and re-run to see the split change.

In [ ]:
rows = []
for folder in find_folders():
    try:
        t, raw, fs = load_line_L(folder)
        bp = bandpass_filter(raw, fs)
        m = mains_metrics(bp, fs)
        key, label = recommend_filter(m["ratio"])
        rows.append(dict(folder=folder, ratio_pct=m["ratio"] * 100,
                         frac_pct=m["frac"] * 100, key=key, label=label))
    except Exception as exc:
        print(f"  skip {folder}: {type(exc).__name__}: {exc}")

table = pd.DataFrame(rows).sort_values("ratio_pct").reset_index(drop=True)
print("recommendation counts:")
print(table["label"].value_counts().to_string())

fig = go.Figure()
for key in ["ff_comb", "iir_comb5"]:
    sub = table[table["key"] == key]
    if sub.empty:
        continue
    fig.add_trace(go.Bar(
        x=sub["ratio_pct"], y=sub["folder"], orientation="h",
        name=sub["label"].iloc[0], marker_color=REC_COLORS[key],
        hovertemplate="%{y}<br>ratio=%{x:.1f}%<extra></extra>"))

fig.add_vline(x=RATIO_THRESHOLD * 100,
              line=dict(color="rgba(0,0,0,0.6)", width=1.5, dash="dash"),
              annotation_text=f"{RATIO_THRESHOLD*100:g}% threshold",
              annotation_position="top", annotation_font_size=11)

fig.update_layout(
    title_text="Mains ratio P(49–51 Hz)/P(rest) across all recordings "
               "(sorted; colour = chosen comb)",
    xaxis_title="ratio (%)  —  log scale", yaxis_title="",
    xaxis_type="log",
    height=max(420, 26 * len(table)), template="plotly_white", barmode="stack",
    legend=dict(orientation="h", y=1.04, x=0.5, xanchor="center"))
fig.update_yaxes(categoryorder="array", categoryarray=table["folder"].tolist())
fig.show()

table[["folder", "ratio_pct", "frac_pct", "label"]]